In [1]:
# ============================================================
# Cell 1: Tools
# Fault-Tolerance Scaffold Simulator
#
# Classical repetition code:
#   1. Encode one logical bit into N physical bits.
#   2. Hardware flips each physical bit with probability p.
#   3. Measurement flips each observed bit with probability q.
#   4. Software decodes by majority vote.
#   5. Count logical failures.
# ============================================================

import random
import math


def make_odd(n):
    """Force N to be a positive odd integer."""
    n = max(1, int(n))
    return n if n % 2 == 1 else n + 1


def majority_decode(bits):
    """Return the majority bit. Assumes odd number of bits."""
    return 1 if sum(bits) > len(bits) / 2 else 0


def one_trial(n, p, q, target=0):
    """
    Run one repetition-code trial.

    p = hardware bit-flip probability
    q = measurement/readout bit-flip probability

    Returns True if logical decoding succeeds, False if it fails.
    """
    n = make_odd(n)

    # Encode target logical bit into N physical bits.
    bits = [target] * n

    # Hardware layer: physical bits may flip.
    physical = [
        1 - b if random.random() < p else b
        for b in bits
    ]

    # Measurement layer: software may observe each bit incorrectly.
    measured = [
        1 - b if random.random() < q else b
        for b in physical
    ]

    # Software layer: majority-vote decoder.
    decoded = majority_decode(measured)

    return decoded == target


def simulate(n=3, p=0.1, q=0.0, trials=10000, target=0, seed=None):
    """
    Monte Carlo estimate of logical failure rate.
    """
    n = make_odd(n)
    p = min(max(float(p), 0.0), 1.0)
    q = min(max(float(q), 0.0), 1.0)
    trials = max(1, int(trials))
    target = int(target)

    if seed is not None:
        random.seed(seed)

    failures = 0

    for _ in range(trials):
        success = one_trial(n=n, p=p, q=q, target=target)
        if not success:
            failures += 1

    return failures / trials


def effective_observed_error(p, q):
    """
    Model A:
    Hardware error p happens first.
    Measurement error q then corrupts what the software observes.

    The software sees an effective bit error rate:

        r = p(1-q) + (1-p)q
          = p + q - 2pq
    """
    return p + q - 2 * p * q


def analytical_logical_failure(n, p, q=0.0):
    """
    Analytical logical failure probability for odd-N repetition code
    with hardware error p and measurement error q.

    Under independent symmetric hardware and measurement errors,
    the software sees effective observed error rate r.
    Logical failure occurs when more than half the measured bits are wrong.
    """
    n = make_odd(n)
    r = effective_observed_error(p, q)

    k_min = n // 2 + 1
    total = 0.0

    for k in range(k_min, n + 1):
        total += math.comb(n, k) * (r ** k) * ((1 - r) ** (n - k))

    return total


def analytical_curve(n, q=0.0, points=101):
    """
    Return p values and analytical logical failure curve for fixed N and q.
    """
    xs = []
    ys = []

    for i in range(points):
        p = i / (points - 1)
        xs.append(p)
        ys.append(analytical_logical_failure(n=n, p=p, q=q))

    return xs, ys


def text_bar(value, width=40):
    """
    Make a simple text bar for notebook display.
    """
    value = min(max(value, 0.0), 1.0)
    filled = int(round(value * width))
    return "█" * filled + "░" * (width - filled)


def print_summary(n, p, q, trials, target, mc_failure, exact_failure):
    """
    Human-readable summary.
    """
    r = effective_observed_error(p, q)

    print("Fault-Tolerance Scaffold Simulator")
    print("=" * 44)
    print()
    print("Architecture:")
    print(f"  Logical bit:                 {target}")
    print(f"  Repetition length N:          {n}")
    print(f"  Correctable measured flips:   0 to {n//2}")
    print(f"  Logical-failure flips:        {n//2 + 1} to {n}")
    print()
    print("Error model:")
    print(f"  Hardware bit-flip error p:    {p:.4f}")
    print(f"  Measurement/readout error q:  {q:.4f}")
    print(f"  Effective observed error r:   {r:.4f}")
    print()
    print("Results:")
    print(f"  Monte Carlo trials:           {trials}")
    print(f"  Monte Carlo logical failure:  {mc_failure:.6f}")
    print(f"  Analytical logical failure:   {exact_failure:.6f}")
    print(f"  Physical hardware error p:    {p:.6f}")
    print()
    print("Logical failure rate:")
    print(f"  {text_bar(mc_failure)} {mc_failure:.4f}")
    print()
    print("Interpretation:")
    print("  Hardware creates physical errors.")
    print("  Measurement errors corrupt what the software sees.")
    print("  The software decoder can only correct what it can infer.")
    print("  Fault tolerance therefore requires hardware/software co-design.")


def print_curve_table(n_values, q=0.0):
    """
    Compact table comparing analytical logical failure at selected p values.
    Useful when avoiding plotting packages.
    """
    p_values = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

    print()
    print("Analytical logical failure probability")
    print(f"Measurement error q = {q:.3f}")
    print("-" * 72)

    header = "N \\ p  " + "".join(f"{p:>10.2f}" for p in p_values)
    print(header)
    print("-" * 72)

    for n in n_values:
        row = f"{n:>5} "
        for p in p_values:
            y = analytical_logical_failure(n=n, p=p, q=q)
            row += f"{y:>10.4f}"
        print(row)

    print("-" * 72)

In [2]:
# ============================================================
# Cell 2: Compact UI, Square Side-by-Side Plots, and Save Figure
#
# Requires Cell 1 functions:
#   simulate()
#   analytical_logical_failure()
#   print_curve_table()
#
# Model:
#   target bit
#   -> hardware bit flip with probability p
#   -> measurement/readout flip with probability q
#   -> majority-vote decoder
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


# ----------------------------
# Small helper functions
# ----------------------------

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def clean_odd_n(n):
    """
    Convert manual N input to a positive odd integer.
    """
    n = int(n)
    if n < 1:
        n = 1
    if n % 2 == 0:
        n += 1
    return n


def add_label_background(labels, alpha=0.88):
    """
    Add a white background box behind contour labels.
    """
    for txt in labels:
        txt.set_bbox(dict(
            facecolor="white",
            edgecolor="none",
            alpha=alpha,
            pad=0.35
        ))


def compact_summary_html(n, p, q, trials, target, mc_failure, exact_failure):
    """
    Compact two-column summary block.
    """
    return f"""
    <div style="
        font-family: Arial, sans-serif;
        max-width: 980px;
        border: 1px solid #ddd;
        border-radius: 8px;
        padding: 12px 14px;
        margin-bottom: 10px;
        background: #fafafa;
    ">
      <div style="display: flex; gap: 36px; align-items: flex-start;">
        <div style="flex: 1;">
          <b>Architecture</b><br>
          String length: <b>N = {n}</b><br>
          Input logical bit: <b>{target}</b><br>
          QEC / decoder: <b>majority vote</b>
        </div>
        <div style="flex: 1;">
          <b>Error model</b><br>
          Hardware bit-flip probability: <b>p = {p:.4f}</b><br>
          Measurement/readout error: <b>q = {q:.4f}</b><br>
          Monte Carlo trials: <b>{trials}</b>
        </div>
      </div>
      <hr style="border: none; border-top: 1px solid #ddd; margin: 10px 0;">
      <div>
        Selected operating point:
        Monte Carlo logical failure = <b>{mc_failure:.6g}</b>;
        analytical logical failure = <b>{exact_failure:.6g}</b>.
      </div>
    </div>
    """


# ----------------------------
# Side-by-side plotting helper
# ----------------------------

def make_side_by_side_plots(
    selected_n=7,
    selected_q=0.0,
    p_max=0.10,
    q_max=0.10,
    target_levels=None,
    grid_points=141,
    log_floor=1e-9,
    save_prefix=None,
    save_png=True,
    save_pdf=True,
    show_plot=True
):
    """
    Create a two-panel figure with square panels.

    Left:
      analytical p -> p_L curves for several N at selected q.

    Right:
      heatmap over p,q for selected N.

    If save_prefix is provided, saves:
      <save_prefix>.png
      <save_prefix>.pdf
    """
    if target_levels is None:
        target_levels = [1e-2, 1e-3, 1e-4]

    target_levels = sorted([
        float(x) for x in target_levels
        if x is not None and float(x) > 0
    ])

    p_max = max(float(p_max), 1e-4)
    q_max = max(float(q_max), 1e-4)

    fig, axes = plt.subplots(
        1, 2,
        figsize=(9.2, 4.6),
        constrained_layout=True
    )

    # ----------------------------
    # Left panel: line plot
    # ----------------------------

    ax = axes[0]

    n_values = [1, 3, 5, 7, 9, 15, 31]
    ps_full = [i / 250 for i in range(251)]

    for n in n_values:
        ys = [
            analytical_logical_failure(n=n, p=p, q=selected_q)
            for p in ps_full
        ]
        ax.plot(ps_full, ys, linewidth=1.35, label=f"N={n}")

    ax.plot(
        ps_full,
        ps_full,
        linestyle="--",
        color="black",
        linewidth=1.1,
        label=r"$p_L=p$"
    )

    ax.axvline(
        0.5,
        linestyle=":",
        color="red",
        linewidth=1.4,
        label=r"$p=0.5$"
    )

    ax.set_title(f"(a) Majority-vote curves, q={selected_q:.2f}", fontsize=10)
    ax.set_xlabel("hardware error p")
    ax.set_ylabel(r"logical failure $p_L$")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_box_aspect(1)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7, loc="upper left", framealpha=0.9)

    # ----------------------------
    # Right panel: heatmap
    # ----------------------------

    ax = axes[1]

    ps = [p_max * i / (grid_points - 1) for i in range(grid_points)]
    qs = [q_max * i / (grid_points - 1) for i in range(grid_points)]

    Z = []
    improvement_boundary = []
    max_value = log_floor

    for q in qs:
        row = []
        improve_row = []

        for p in ps:
            val = analytical_logical_failure(n=selected_n, p=p, q=q)
            val_for_log = max(val, log_floor)

            row.append(val_for_log)
            improve_row.append(val - p)
            max_value = max(max_value, val_for_log)

        Z.append(row)
        improvement_boundary.append(improve_row)

    vmin = min(log_floor, min(target_levels) if target_levels else log_floor)
    vmax = min(1.0, max(1e-2, max_value))

    im = ax.imshow(
        Z,
        origin="lower",
        extent=[0, p_max, 0, q_max],
        aspect="auto",
        norm=LogNorm(vmin=vmin, vmax=vmax)
    )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(r"$p_L$", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    # Red logical-error target contours.
    visible_targets = [
        level for level in target_levels
        if level >= vmin and level <= max_value
    ]

    if visible_targets:
        cs = ax.contour(
            ps,
            qs,
            Z,
            levels=visible_targets,
            colors="red",
            linewidths=1.7
        )

        labels = ax.clabel(
            cs,
            inline=True,
            fontsize=10,
            fmt={level: f"{level:.0e}" for level in visible_targets}
        )
        add_label_background(labels)

    # Black dashed improvement boundary p_L = p.
    # Manual label placement keeps the label away from the y-axis
    # when the heatmap view is zoomed.
    try:
        cs2 = ax.contour(
            ps,
            qs,
            improvement_boundary,
            levels=[0.0],
            colors="black",
            linewidths=1.5,
            linestyles="--"
        )

        labels2 = ax.clabel(
            cs2,
            inline=True,
            fontsize=10,
            fmt={0.0: r"$p_L=p$"},
            manual=[(0.60 * p_max, 0.60 * q_max)]
        )
        add_label_background(labels2)

    except Exception:
        pass

    # Only show p=0.5 if user zooms out far enough.
    if p_max >= 0.5:
        ax.axvline(
            0.5,
            color="red",
            linestyle=":",
            linewidth=1.5
        )

    ax.set_title(f"(b) Operating region, N={selected_n}", fontsize=10)
    ax.set_xlabel("hardware error p")
    ax.set_ylabel("measurement error q")
    ax.set_xlim(0, p_max)
    ax.set_ylim(0, q_max)
    ax.set_box_aspect(1)

    if save_prefix:
        if save_png:
            fig.savefig(f"{save_prefix}.png", dpi=300, bbox_inches="tight")
        if save_pdf:
            fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight")

    if show_plot:
        plt.show()
    else:
        plt.close(fig)

    return fig


# ----------------------------
# UI controls
# ----------------------------

# Wider widget styles.
main_style = {"description_width": "95px"}
target_style = {"description_width": "165px"}
save_style = {"description_width": "110px"}

main_layout = widgets.Layout(width="260px")
target_layout = widgets.Layout(width="390px")
save_layout = widgets.Layout(width="320px")

# Main manual controls.
n_input = widgets.BoundedIntText(
    value=7,
    min=1,
    max=999,
    step=2,
    description="N bits",
    continuous_update=False,
    style=main_style,
    layout=main_layout
)

p_input = widgets.BoundedFloatText(
    value=0.05,
    min=0.0,
    max=1.0,
    step=0.01,
    description="p",
    continuous_update=False,
    style=main_style,
    layout=main_layout
)

q_input = widgets.BoundedFloatText(
    value=0.01,
    min=0.0,
    max=0.5,
    step=0.01,
    description="q",
    continuous_update=False,
    style=main_style,
    layout=main_layout
)

trials_input = widgets.BoundedIntText(
    value=10000,
    min=1,
    max=10000000,
    step=1000,
    description="trials",
    continuous_update=False,
    style=main_style,
    layout=main_layout
)

target_dropdown = widgets.Dropdown(
    options=[0, 1],
    value=0,
    description="target",
    style=main_style,
    layout=main_layout
)

seed_box = widgets.IntText(
    value=7,
    description="seed",
    style=main_style,
    layout=main_layout
)

use_seed_box = widgets.Checkbox(
    value=True,
    description="use seed",
    layout=main_layout
)

# Heatmap axis controls.
pmax_input = widgets.BoundedFloatText(
    value=0.10,
    min=0.001,
    max=1.0,
    step=0.01,
    description="p max",
    continuous_update=False,
    style=main_style,
    layout=main_layout
)

qmax_input = widgets.BoundedFloatText(
    value=0.10,
    min=0.001,
    max=0.5,
    step=0.01,
    description="q max",
    continuous_update=False,
    style=main_style,
    layout=main_layout
)

# User-defined logical-error target contours.
target1_input = widgets.FloatLogSlider(
    value=1e-2,
    base=10,
    min=-9,
    max=-1,
    step=1,
    description="logical error target 1",
    readout_format=".0e",
    continuous_update=False,
    style=target_style,
    layout=target_layout
)

target2_input = widgets.FloatLogSlider(
    value=1e-3,
    base=10,
    min=-9,
    max=-1,
    step=1,
    description="logical error target 2",
    readout_format=".0e",
    continuous_update=False,
    style=target_style,
    layout=target_layout
)

target3_input = widgets.FloatLogSlider(
    value=1e-4,
    base=10,
    min=-9,
    max=-1,
    step=1,
    description="logical error target 3",
    readout_format=".0e",
    continuous_update=False,
    style=target_style,
    layout=target_layout
)

# Save controls.
save_prefix_input = widgets.Text(
    value="fault_tolerance_scaffold_figure",
    description="save prefix",
    style=save_style,
    layout=save_layout
)

save_png_box = widgets.Checkbox(
    value=True,
    description="save PNG",
    layout=save_layout
)

save_pdf_box = widgets.Checkbox(
    value=True,
    description="save PDF",
    layout=save_layout
)

run_button = widgets.Button(
    description="Run simulator",
    button_style="primary",
    layout=widgets.Layout(width="160px")
)

output = widgets.Output()


# ----------------------------
# Main callback
# ----------------------------

def run_demo(_=None):
    with output:
        clear_output(wait=True)

        n = clean_odd_n(n_input.value)
        p = clamp(float(p_input.value), 0.0, 1.0)
        q = clamp(float(q_input.value), 0.0, 0.5)
        trials = max(1, int(trials_input.value))
        target = target_dropdown.value
        seed = seed_box.value if use_seed_box.value else None

        p_max = clamp(float(pmax_input.value), 0.001, 1.0)
        q_max = clamp(float(qmax_input.value), 0.001, 0.5)

        target_levels = sorted([
            float(target1_input.value),
            float(target2_input.value),
            float(target3_input.value)
        ])

        save_prefix = save_prefix_input.value.strip()
        save_prefix = save_prefix if save_prefix else None

        # Keep the UI honest if user entered an even N.
        n_input.value = n

        mc_failure = simulate(
            n=n,
            p=p,
            q=q,
            trials=trials,
            target=target,
            seed=seed
        )

        exact_failure = analytical_logical_failure(
            n=n,
            p=p,
            q=q
        )

        display(HTML(compact_summary_html(
            n=n,
            p=p,
            q=q,
            trials=trials,
            target=target,
            mc_failure=mc_failure,
            exact_failure=exact_failure
        )))

        make_side_by_side_plots(
            selected_n=n,
            selected_q=q,
            p_max=p_max,
            q_max=q_max,
            target_levels=target_levels,
            grid_points=141,
            log_floor=1e-9,
            save_prefix=save_prefix,
            save_png=save_png_box.value,
            save_pdf=save_pdf_box.value,
            show_plot=True
        )

        if save_prefix:
            saved = []
            if save_png_box.value:
                saved.append(f"{save_prefix}.png")
            if save_pdf_box.value:
                saved.append(f"{save_prefix}.pdf")

            if saved:
                display(HTML(
                    "<div style='font-family: Arial; margin: 8px 0;'>"
                    f"<b>Saved figure:</b> {', '.join(saved)}"
                    "</div>"
                ))

        display(HTML("""
        <div style="
            font-family: Arial, sans-serif;
            max-width: 980px;
            margin-top: 8px;
            padding: 10px 12px;
            border-left: 4px solid #555;
            background: #f7f7f7;
            line-height: 1.35;
        ">
        <b>Interpretation.</b>
        The line plot shows the broad majority-vote threshold behavior:
        at <b>p = 0.5</b>, each bit is random and majority vote has no information.
        The heatmap is the engineering view: for fixed <b>N</b>, it shows which
        combinations of hardware error <b>p</b> and measurement error <b>q</b>
        reach the selected logical-error targets.
        Red contours mark target logical failure rates; the black dashed contour
        marks <b>p<sub>L</sub> = p</b>, where encoding stops improving over one physical bit.
        </div>
        """))

        display(HTML(
            "<div style='font-family: Arial; margin-top: 6px;'>"
            "<b>Logical-error target contours used:</b> "
            + ", ".join([f"{x:.0e}" for x in target_levels])
            + "</div>"
        ))

        print_curve_table(
            n_values=[1, 3, 5, 7, 9, 15, 31],
            q=q
        )


run_button.on_click(run_demo)


# ----------------------------
# Layout
# ----------------------------

main_controls = widgets.VBox([
    widgets.HTML("<b>Selected operating point</b>"),
    n_input,
    p_input,
    q_input,
    trials_input
])

run_controls = widgets.VBox([
    widgets.HTML("<b>Run settings</b>"),
    target_dropdown,
    use_seed_box,
    seed_box,
    run_button
])

heatmap_controls = widgets.VBox([
    widgets.HTML("<b>Heatmap view</b>"),
    pmax_input,
    qmax_input,
    target1_input,
    target2_input,
    target3_input
])

save_controls = widgets.VBox([
    widgets.HTML("<b>Save figure</b>"),
    save_prefix_input,
    save_png_box,
    save_pdf_box
])

ui_top = widgets.HBox([
    main_controls,
    run_controls,
    heatmap_controls
])

ui_bottom = widgets.HBox([
    save_controls
])

display(HTML("""
<div style="font-family: Arial, sans-serif; max-width: 980px;">
<h3>Fault-Tolerance Scaffold Simulator</h3>
<p>
Classical repetition code with hardware error <b>p</b>, measurement error <b>q</b>,
and majority-vote decoding. The line plot shows the broad threshold idea;
the heatmap zooms into the useful operating region. The side-by-side figure
is saved as PNG/PDF for inclusion in the LaTeX brief.
</p>
</div>
"""))

display(ui_top)
display(ui_bottom)
display(output)

run_demo()

Output()